## This is the code to train the model and acquire influence for Number of Features Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 10 features in total. The separation is set to 1 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 5000, and the test size is 500. The number of features is set to 10 plus the 42 noise features. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, the only thing changing here is about modifying the noise features added. This will be explained in this code. For the other common information, Please refer to the base code for more detailed explanation.**

**Guideline**:  
Read in / Construct Datasets -> **Choose the Noise Feature Size** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Noise Feature Size and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [136]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [137]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [138]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [139]:
import random
from keras.optimizers import SGD

In [140]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [141]:
from sklearn.datasets import make_classification

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 10 features with binary classification problems. The later options will turn that into a 10 features + 42 noise features, 8000 train set and 500 test set sample. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the noise features. In this experiment, all the other things are fixed, but the number of noise features is changing to test on different number of features.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. train_sizes determines the later subset data. In this experiment, the sample size will always be set to 5000. Sep to make sure the dataset is distinguishable.

In [142]:
train_pool = 16000
test_size = 500
train_sizes=[1000,2000,3000,4000,5000,6000,7000,8000,9000,10000]
n_features=10
seed=42
sep = 1

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [143]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_clusters_per_class = 1,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

3. **The Feature Noises are added at this step.** n_noise determines the total number of noise features generated. The k from Ks will be the exact noise features added to the dataset during each run. **Change KS[i] to modify the noise features.**

In [144]:
KS = [10,14,18,22,26,30,34,38,42]
n_noise = 90
rng = np.random.RandomState(seed) 
X_noise = rng.normal(loc=0.0, scale=1.0, size=(X.shape[0], n_noise))

k= KS[8]

X = np.hstack([X, X_noise[:, :k]])
# k = 0

4. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [145]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features+k)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.462785   1.784031   2.750230   1.983940   0.331680   1.174118   
1      -3.425860   3.091771  -0.996399   1.152936   0.487537  -0.794771   
2      -3.666989   1.969536   1.389431   2.247002   2.385491   0.830315   
3      -4.224882  -1.480716   3.421315   1.025147   0.097334   4.042792   
4      -4.244179  -5.092764   3.780899  -3.046639  -3.902753  -3.343005   
...          ...        ...        ...        ...        ...        ...   
16495  -1.738303  -2.368518   0.184498  -2.834851  -1.781499   0.763730   
16496  -2.446645   0.955477   4.436585  -0.664160   0.935989  -4.083520   
16497  -2.289426   0.550712   2.845396   3.849258  -0.076908   2.120920   
16498  -5.918908   2.077796   2.784047   1.993381  -4.229283   4.286199   
16499  -2.527862  -1.756233   3.466682  -5.149739   2.350795   0.277100   

       feature_7  feature_8  feature_9  feature_10  ...  feature_45  \
0      -2.195223  -1.469477 

In [146]:
df_train_pool = df.iloc[:train_pool].reset_index(drop=True)
df_test = df.iloc[train_pool:].reset_index(drop=True)

In [147]:
print(df_train_pool.head())
print(df_test.head())

   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0  -2.462785   1.784031   2.750230   1.983940   0.331680   1.174118   
1  -3.425860   3.091771  -0.996399   1.152936   0.487537  -0.794771   
2  -3.666989   1.969536   1.389431   2.247002   2.385491   0.830315   
3  -4.224882  -1.480716   3.421315   1.025147   0.097334   4.042792   
4  -4.244179  -5.092764   3.780899  -3.046639  -3.902753  -3.343005   

   feature_7  feature_8  feature_9  feature_10  ...  feature_45  feature_46  \
0  -2.195223  -1.469477   1.106515    0.137798  ...    0.822545   -1.220844   
1   2.273718   2.553062   1.129106   -5.716311  ...    0.586857    2.190456   
2   1.854461   3.752189   1.141437   -4.276726  ...   -0.315269    0.758969   
3  -2.073007  -2.305776   0.674058   -0.978083  ...   -0.020902    0.117327   
4  -3.954134   1.157540   1.570334   -3.043344  ...    1.179440   -0.469176   

   feature_47  feature_48  feature_49  feature_50  feature_51  feature_52  \
0    0.208864   -1.95

5. Here, we choose the subset of the full dataset. By setting features_to_test, we have the subset feature size. By changing nested_train_dfs, we have different sample sizes. Here the feature size will always be n_features+k since we use all the features plus noise features to form the subset dataset.

In [148]:
features_to_test = n_features+k
selected_features = [f'feature_{i+1}' for i in range(features_to_test)]

nested_train_dfs = [df_train_pool.iloc[:size].reset_index(drop=True) for size in train_sizes]
nested_train_dfs = [df[ selected_features + ['label', 'id'] ].copy()for df in nested_train_dfs]

df_test = df_test[ selected_features + ['label', 'id'] ].copy()

core_1000_ids = nested_train_dfs[0]['id'].tolist()

In [149]:
train_df = nested_train_dfs[9]

In [150]:
train_df.to_csv("train_df_feature52.csv", index=False)